# 🎵 K-Pop Dance Analyzer
### วิเคราะห์และเปรียบเทียบท่าเต้นของคุณกับศิลปิน K-pop

**วิธีใช้งาน:**
1. ติดตั้ง dependencies (Cell 1)
2. ใส่ YouTube URL ของ K-pop MV ที่ต้องการ (Cell 3)
3. ใส่ path ของวิดีโอตัวเองที่เต้นแล้ว (Cell 3)
4. Run ทุก cell ตามลำดับ

---
**ข้อแนะนำสำหรับวิดีโอของตัวเอง:**
- 🎥 ถ่ายให้เห็นทั้งร่างกาย head to toe
- 💡 แสงสว่างเพียงพอ ไม่ย้อนแสง
- 📐 ถ่ายจากมุมตรง ไม่เอียงมาก
- ⏱️ ความยาว 15 วินาที - 2 นาที เหมาะที่สุด

In [ ]:
# Cell 1: ติดตั้ง dependencies
# (รันครั้งแรกครั้งเดียว)
import subprocess, sys
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "-r", "requirements.txt",
    "--break-system-packages", "-q"
])
print("✅ ติดตั้ง dependencies เสร็จแล้ว!")

In [ ]:
# Cell 2: Import libraries
import os, importlib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Image, Video, HTML
import importlib, reference_processor
importlib.reload(reference_processor)

# reload เสมอเพื่อรับโค้ดใหม่จากไฟล์
import reference_processor, dance_comparator
importlib.reload(reference_processor)
importlib.reload(dance_comparator)

from reference_processor import (
    download_youtube_video,
    extract_pose_from_video,
    preview_person_selection,
    create_labeled_video,
    create_person_clip,
    save_pose_data,
    load_pose_data,
)
from dance_comparator import compare_dance, quick_compare, grade_score

print('✅ Import สำเร็จ! พร้อมใช้งาน')

## ⚙️ ตั้งค่า
แก้ไข cell นี้ตามความต้องการ

In [ ]:
# Cell 3: ตั้งค่าหลัก

# === ตั้งค่า Reference Video (K-pop Artist) ===
YOUTUBE_URL = "https://www.youtube.com/watch?v=Ba-y6OgrC70"
# "https://youtube.com/shorts/1TSjeNQFknI?feature=share"   # UP Dance Practice (non-mirrored)
START_TIME  = "0:00"
END_TIME    = "0:58"

# === เลือกคนที่ต้องการ track ===
PERSON_INDEX = None   # 👈 แก้ตรงนี้หลังดู preview

# === วิดีโอของคุณ ===
USER_VIDEO       = "https://youtu.be/FQN-1IbVKMA?si=yFHxE3ZtvnHYdq0B"   # 👈 ใส่ path หรือ YouTube URL
USER_START_TIME  = "0:00"   # เช่น "0:30" หรือ None = ตั้งแต่ต้น
USER_END_TIME    = "0:58"   # เช่น "1:30" หรือ None = จนจบ

# === Sync offset (ดูจาก Cell 6.5) ===
REF_START_OFFSET = 0.21   # 👈 ถ้า user เริ่มเร็วกว่า ref → ใส่ค่าจาก Cell 6.5
USER_START_OFFSET = 0.0   # 👈 ถ้า user เริ่มช้ากว่า ref → ใส่ค่าจาก Cell 6.5
PASS_THRESHOLD    = 0.7   # 👈 เกณฑ์ PASS (0.0-1.0) — สูง=เข้มงวด, ต่ำ=ผ่อนปรน

user_video_path = USER_VIDEO

# === ตั้งค่าอื่นๆ ===
SAMPLE_FPS = 10
OUTPUT_DIR = "results"

print(f'Reference: {YOUTUBE_URL}  [{START_TIME} → {END_TIME}]')
print(f'Person index: {PERSON_INDEX}')
print(f'วิดีโอของคุณ: {USER_VIDEO}')
print(f'REF offset : {REF_START_OFFSET}s  (user เร็วกว่า)')
print(f'USER offset: {USER_START_OFFSET}s  (user ช้ากว่า)')
print(f'PASS threshold: {PASS_THRESHOLD}  (เกณฑ์ตัดสิน)')


## 📥 Step 1: ดาวน์โหลด Reference Video

In [ ]:
# Cell 4a: ดาวน์โหลดวิดีโอ reference จาก YouTube
import importlib, reference_processor
importlib.reload(reference_processor)
from reference_processor import download_youtube_video

ref_video_path = download_youtube_video(
    url=YOUTUBE_URL,
    output_dir=os.path.join(OUTPUT_DIR, "reference_videos"),
    start_time=START_TIME,
    end_time=END_TIME,
)
print(f'✅ ดาวน์โหลดสำเร็จ: {ref_video_path}')

### 👥 Cell 4b: ดูว่ามีกี่คนในวิดีโอ (สำหรับวิดีโอที่มีนักเต้นหลายคน)
รัน cell นี้เพื่อดูภาพ preview พร้อมกรอบสีรอบแต่ละคน
แล้วนำหมายเลขที่ต้องการกลับไปใส่ `PERSON_INDEX` ใน Cell 3

In [ ]:
# Cell 4b: คลิกเลือกคนที่ต้องการ track
# หน้าต่างจะเปิดขึ้นมา → คลิกที่คนที่ต้องการ → ปิดหน้าต่าง
os.makedirs(OUTPUT_DIR, exist_ok=True)
preview_path = os.path.join(OUTPUT_DIR, 'person_preview.png')

selected_index = preview_person_selection(
    video_path=ref_video_path,
    save_path=preview_path,
)

# อัปเดต PERSON_INDEX อัตโนมัติ
PERSON_INDEX = selected_index
print(f'✅ PERSON_INDEX ถูกตั้งเป็น {PERSON_INDEX} แล้ว รัน Cell 5 ได้เลยครับ')

In [ ]:
# Cell 4c: ดูวิดีโอที่มีกรอบหมายเลขทุกคน → ดูว่า PERSON_INDEX ไหนคือคนที่ต้องการ
import importlib, reference_processor
importlib.reload(reference_processor)
from reference_processor import create_labeled_video
from IPython.display import Video

labeled_path = os.path.join(OUTPUT_DIR, "labeled_preview.mp4")
create_labeled_video(video_path=ref_video_path, output_path=labeled_path)

print(f"✅ ดูวิดีโอแล้วจด PERSON_INDEX ของคนที่ต้องการ")
display(Video(labeled_path, width=800, embed=True))

## 🦴 Step 2: Extract Pose จาก Reference Video

In [ ]:
# Cell 5: Extract pose keypoints จาก reference
# (ถ้ามีหลายคน ให้แก้ PERSON_INDEX ใน Cell 3 ก่อนรัน cell นี้)
print('กำลัง extract pose จาก reference video...')
if PERSON_INDEX is not None:
    print(f'→ โหมด Multi-person: จะ track คนที่ {PERSON_INDEX} (นับจาก 0 = ซ้ายสุด)')

ref_data = extract_pose_from_video(
    ref_video_path,
    sample_fps=SAMPLE_FPS,
    person_index=PERSON_INDEX,   # None = คนเดียว, 0/1/2/... = เลือกคน
)
ref_data['video_info']['path'] = ref_video_path

os.makedirs(os.path.join(OUTPUT_DIR, 'pose_cache'), exist_ok=True)
save_pose_data(ref_data, os.path.join(OUTPUT_DIR, 'pose_cache', 'reference'))

print(f'\n✅ Extract สำเร็จ: {len(ref_data["keypoints"])} frames')
print(f'   Duration: {ref_data["video_info"]["duration"]:.1f}s')

In [ ]:
# Cell 6: ดู sample poses ของ reference (optional)
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
fig.patch.set_facecolor('#1a1a2e')

kps_list = ref_data['keypoints']
sample_indices = np.linspace(0, len(kps_list)-1, 5, dtype=int)

for ax, idx in zip(axes, sample_indices):
    kp = kps_list[idx]
    ax.set_facecolor('#16213e')
    ax.scatter(kp[:, 0], -kp[:, 1], c='#00d4ff', s=30, zorder=5)
    
    # วาด connections
    connections = [(0,1),(0,2),(1,3),(2,4),(3,5),(4,6),(1,7),(2,8),(7,8),(7,9),(8,10),(9,11),(10,12)]
    for i, j in connections:
        if i < len(kp) and j < len(kp):
            ax.plot([kp[i,0], kp[j,0]], [-kp[i,1], -kp[j,1]], 
                   color='#ff6b9d', linewidth=1.5, alpha=0.8)
    
    ts = ref_data['timestamps'][idx]
    ax.set_title(f't={ts:.1f}s', color='white', fontsize=9)
    ax.axis('off')

plt.suptitle('Sample Poses - Reference (Artist)', color='white', fontsize=12)
plt.tight_layout()
plt.show()
print('👆 ท่าทางของศิลปินที่ extract ได้')

## 🕺 Step 3: เปรียบเทียบท่าเต้น

### 🎵 Cell 6.5: วิเคราะห์ Audio Offset
ตรวจสอบว่า user video เริ่มช้ากว่า reference กี่วินาที
ก่อนดู comparison video จะได้รู้ว่าที่ดูช้าเพราะ **คลิปเริ่มช้า** หรือ **เต้นช้าจริงๆ**

In [ ]:
# Cell 6.5: วิเคราะห์ Audio Offset
import subprocess, tempfile, os
import numpy as np

user_video_path = USER_VIDEO  # จาก Cell 3

def extract_audio_wav(video_path, out_wav, sr=22050):
    """Extract audio — รองรับทั้ง local file และ YouTube URL"""
    is_url = video_path.startswith('http')

    if is_url:
        # ใช้ yt-dlp download audio โดยตรง
        try:
            r = subprocess.run([
                'yt-dlp', '-x', '--audio-format', 'wav',
                '--audio-quality', '5',
                '--postprocessor-args', f'-ar {sr} -ac 1',
                '-o', out_wav.replace('.wav', '.%(ext)s'),
                video_path
            ], capture_output=True, text=True)
            # yt-dlp อาจบันทึกเป็น .wav หรือ rename
            if os.path.exists(out_wav) and os.path.getsize(out_wav) > 1000:
                return True
            # หาไฟล์ที่ดาวน์โหลดมา
            base = out_wav.replace('.wav', '')
            for ext in ['.wav', '.m4a', '.webm', '.mp3']:
                if os.path.exists(base + ext):
                    # แปลงเป็น wav ด้วย ffmpeg
                    subprocess.run(['ffmpeg', '-y', '-i', base + ext,
                                    '-ar', str(sr), '-ac', '1', out_wav],
                                   capture_output=True)
                    return os.path.exists(out_wav)
            print(f'  yt-dlp stderr: {r.stderr[-200:]}')
            return False
        except FileNotFoundError:
            print('  ⚠️ yt-dlp ไม่พบ')
            return False
    else:
        # local file → ffmpeg
        try:
            r = subprocess.run([
                'ffmpeg', '-y', '-i', video_path,
                '-vn', '-acodec', 'pcm_s16le', '-ar', str(sr), '-ac', '1', out_wav
            ], capture_output=True, text=True)
            return os.path.exists(out_wav) and os.path.getsize(out_wav) > 1000
        except FileNotFoundError:
            print('  ⚠️ ffmpeg ไม่พบ ลองติดตั้ง: brew install ffmpeg')
            return False

def compute_audio_offset(ref_video, user_video):
    import librosa
    from scipy.signal import correlate

    with tempfile.TemporaryDirectory() as tmpdir:
        ref_wav  = os.path.join(tmpdir, 'ref.wav')
        user_wav = os.path.join(tmpdir, 'user.wav')

        print('🎵 Extracting audio from reference...')
        if not extract_audio_wav(ref_video, ref_wav):
            print('❌ ไม่สามารถ extract audio จาก reference')
            return None

        print('🎵 Extracting audio from user video...')
        if not extract_audio_wav(user_video, user_wav):
            print('❌ ไม่สามารถ extract audio จาก user video')
            return None

        sr = 22050
        y_ref,  _ = librosa.load(ref_wav,  sr=sr)
        y_user, _ = librosa.load(user_wav, sr=sr)

        print(f'  Ref audio : {len(y_ref)/sr:.1f}s')
        print(f'  User audio: {len(y_user)/sr:.1f}s')

        env_ref  = librosa.onset.onset_strength(y=y_ref,  sr=sr)
        env_user = librosa.onset.onset_strength(y=y_user, sr=sr)

        corr = correlate(env_ref, env_user, mode='full')
        hop_length = 512
        lag_frames = np.argmax(corr) - (len(env_user) - 1)
        offset_sec = lag_frames * hop_length / sr

        return offset_sec

# รัน
print(f'Reference : {ref_video_path}')
print(f'User video: {user_video_path}')
print()

offset = compute_audio_offset(ref_video_path, user_video_path)

if offset is not None:
    abs_offset = abs(offset)
    print()
    print('=' * 50)
    if abs_offset < 0.1:
        print(f'✅ เริ่มพร้อมกัน (offset = {offset:.2f}s)')
        print('   ถ้าเต้นช้า → ช้าจริงๆ ไม่ใช่แค่เริ่มช้า')
    elif offset > 0:
        print(f'⏱ User video เริ่มช้ากว่า {abs_offset:.2f} วินาที')
        print(f'   → แก้: ใส่ USER_START_OFFSET = {abs_offset:.2f} ใน Cell 3')
        print(f'   (ข้าม {abs_offset:.2f}s แรกของ user video เพื่อ sync)')
    else:
        print(f'⏱ User video เริ่มเร็วกว่า {abs_offset:.2f} วินาที')
        print(f'   → แก้: ใส่ REF_START_OFFSET = {abs_offset:.2f} ใน Cell 3')
        print(f'   (ข้าม {abs_offset:.2f}s แรกของ reference เพื่อ sync)')
    print('=' * 50)


In [ ]:
# Cell 7: เปรียบเทียบท่าเต้น

# ถ้า USER_VIDEO เป็น YouTube URL → download ก่อน
user_video_path = USER_VIDEO
if USER_VIDEO and (USER_VIDEO.startswith("http") or "youtu" in USER_VIDEO):
    print("🔗 พบ YouTube URL → กำลัง download วิดีโอของคุณ...")
    user_video_path = download_youtube_video(
        url=USER_VIDEO,
        output_dir=os.path.join(OUTPUT_DIR, "user_videos"),
        start_time=USER_START_TIME,
        end_time=USER_END_TIME,
    )
    print(f"✅ download เสร็จ: {user_video_path}")

if not os.path.exists(user_video_path):
    print(f'❌ ไม่พบวิดีโอ: {user_video_path}')
    print('   กรุณาใส่ path หรือ YouTube URL ที่ถูกต้องใน USER_VIDEO')
else:
    ref_offset  = REF_START_OFFSET  if 'REF_START_OFFSET'  in dir() else 0.0
    user_offset = USER_START_OFFSET if 'USER_START_OFFSET' in dir() else 0.0
    results = compare_dance(
        reference_data=ref_data,
        user_video_path=user_video_path,
        output_dir=OUTPUT_DIR,
        sample_fps=SAMPLE_FPS,
        create_comparison_video=True,
        ref_start_offset=ref_offset,
        user_start_offset=user_offset,
        pass_threshold=PASS_THRESHOLD if 'PASS_THRESHOLD' in dir() else 0.5,
    )
    print('✅ วิเคราะห์เสร็จแล้ว!')


In [ ]:
# Cell 7b: Anomaly Detection — Pass/Fail ทุก Frame
# ต้องรัน dance_EDA.ipynb Section 8 ก่อน
if 'results' not in dir():
    print('กรุณารัน Cell 7 ก่อน')
else:
    pf = results.get('pf_analysis', {})
    if not pf.get('available'):
        print('⚠️  Anomaly Detection model ยังไม่ถูกโหลด')
        print('   → รัน dance_EDA.ipynb Section 8 ให้ครบก่อน แล้ว Restart Kernel')
    else:
        import matplotlib.pyplot as plt
        import matplotlib.patches as mpatches
        import numpy as np

        frames    = pf['frames']
        pass_rate = pf['pass_rate']
        thresh    = results.get('pass_threshold', 0.5)
        times     = np.array([f['time']      for f in frames])
        probs     = np.array([f['pass_prob'] for f in frames])
        labels    = np.array([f['is_pass']   for f in frames])

        fig = plt.figure(figsize=(16, 8))
        fig.patch.set_facecolor('#0f0f1a')
        gs = fig.add_gridspec(3, 1, height_ratios=[0.6, 1.4, 1], hspace=0.45)

        # ── Row 1: Frame-by-frame color strip ──────────────────────
        ax_strip = fig.add_subplot(gs[0])
        ax_strip.set_facecolor('#1a1a2e')
        colors_strip = ['#2ecc71' if p else '#e74c3c' for p in labels]
        for i, (t, c) in enumerate(zip(times, colors_strip)):
            dt = times[1] - times[0] if len(times) > 1 else 0.1
            ax_strip.barh(0, dt, left=t, height=1.0, color=c, linewidth=0)
        ax_strip.set_xlim(times[0], times[-1])
        ax_strip.set_yticks([])
        ax_strip.set_title(
            f'Anomaly Detection — Per Frame  |  Overall PASS: {pass_rate:.1%}  |  Threshold: {thresh}',
            color='white', fontsize=12, pad=6)
        for sp in ax_strip.spines.values(): sp.set_visible(False)
        patch_p = mpatches.Patch(color='#2ecc71', label='PASS (Normal)')
        patch_f = mpatches.Patch(color='#e74c3c', label='FAIL (Abnormal)')
        ax_strip.legend(handles=[patch_p, patch_f], loc='upper right',
                        facecolor='#1a1a2e', labelcolor='white', fontsize=9)

        # ── Row 2: Normality probability curve ─────────────────────
        ax_curve = fig.add_subplot(gs[1])
        ax_curve.set_facecolor('#1a1a2e')
        ax_curve.fill_between(times, probs, thresh,
            where=probs >= thresh, color='#2ecc71', alpha=0.35, interpolate=True)
        ax_curve.fill_between(times, probs, thresh,
            where=probs < thresh,  color='#e74c3c', alpha=0.35, interpolate=True)
        ax_curve.plot(times, probs, color='white', linewidth=0.8, alpha=0.7)
        ax_curve.axhline(thresh, color='#ffff00', linestyle='--', linewidth=1,
                          alpha=0.6, label=f'Threshold {thresh:.0%}')
        ax_curve.set_xlim(times[0], times[-1])
        ax_curve.set_ylim(-0.05, 1.15)
        ax_curve.set_ylabel('Normality Score', color='#aaa', fontsize=10)
        ax_curve.set_xlabel('Time (s)', color='#aaa', fontsize=10)
        ax_curve.set_title('Normality Score — High = PASS, Low = FAIL',
                            color='white', fontsize=11)
        ax_curve.tick_params(colors='#888')
        ax_curve.yaxis.set_major_formatter(plt.FuncFormatter(lambda y,_: f'{y:.0%}'))
        ax_curve.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=9)
        for sp in ['top','right']: ax_curve.spines[sp].set_visible(False)
        for sp in ['bottom','left']: ax_curve.spines[sp].set_color('#333')

        # ── Row 3: FAIL runs ────────────────────────────────────────
        ax_fail = fig.add_subplot(gs[2])
        ax_fail.set_facecolor('#1a1a2e')
        ax_fail.axis('off')

        fail_runs = []
        in_fail = False
        for f in frames:
            if not f['is_pass'] and not in_fail:
                run_start = f['time']; in_fail = True
            elif f['is_pass'] and in_fail:
                fail_runs.append((run_start, f['time'])); in_fail = False
        if in_fail:
            fail_runs.append((run_start, frames[-1]['time']))
        fail_runs = [(s, e) for s, e in fail_runs if e - s >= 1.0]

        ax_fail.set_title('Sections Needing Practice (Abnormal ≥ 1s)', color='white', fontsize=11)
        if fail_runs:
            lines = []
            for i, (s, e) in enumerate(fail_runs[:10]):
                ms, ss = divmod(int(s), 60)
                me, se = divmod(int(e), 60)
                lines.append(f"{i+1:2d}.  {ms:02d}:{ss:02d} – {me:02d}:{se:02d}  ({e-s:.1f}s)")
            if len(fail_runs) > 10:
                lines.append(f"    ... และอีก {len(fail_runs)-10} ช่วง")
            ax_fail.text(0.02, 0.85, '\n'.join(lines),
                transform=ax_fail.transAxes, va='top', ha='left',
                fontsize=10, color='#ff6b6b', fontfamily='monospace')
        else:
            ax_fail.text(0.5, 0.5, '🎉 No abnormal sections longer than 1 second!',
                transform=ax_fail.transAxes, va='center', ha='center',
                fontsize=13, color='#2ecc71')

        plt.savefig(f'{OUTPUT_DIR}/anomaly_timeline.png',
                    bbox_inches='tight', facecolor=fig.get_facecolor(), dpi=120)
        plt.show()

        print(f'\n📊 Summary:')
        print(f'   Total frames : {len(frames):,}  ({times[-1]:.1f}s)')
        print(f'   PASS frames  : {pf["pass_count"]:,}  ({pass_rate:.1%})')
        print(f'   FAIL frames  : {pf["fail_count"]:,}  ({1-pass_rate:.1%})')
        print(f'   FAIL runs    : {len(fail_runs)} ช่วง  (≥ 1s)')


## 📊 Step 4: ดูผลลัพธ์

In [ ]:
# Cell 8: แสดงกราฟวิเคราะห์
if 'results' in dir() and results.get('output_plot'):
    img = mpimg.imread(results['output_plot'])
    plt.figure(figsize=(14, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('กรุณารัน Cell 7 ก่อน')

In [ ]:
# Cell 9: Body Part Analysis Chart
if 'results' in dir():
    body = results['body_analysis']
    parts = list(body.keys())
    scores = [body[p]['score'] for p in parts]
    grades = [body[p]['grade'] for p in parts]

    colors = []
    for s in scores:
        if s >= 0.8:   colors.append('#00cc44')
        elif s >= 0.6: colors.append('#ffcc00')
        else:          colors.append('#ff4444')

    fig, ax = plt.subplots(figsize=(8, 5))
    fig.patch.set_facecolor('#1a1a2e')
    ax.set_facecolor('#16213e')

    bars = ax.barh(parts, scores, color=colors, height=0.5, edgecolor='none')

    for bar, score, grade in zip(bars, scores, grades):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{score:.0%} ({grade})', va='center', color='white', fontsize=11)

    ax.axvline(x=0.8, color='#00cc44', linestyle='--', alpha=0.5, linewidth=1)
    ax.axvline(x=0.6, color='#ffcc00', linestyle='--', alpha=0.5, linewidth=1)

    ax.set_xlim(0, 1.2)
    ax.set_title('Body Part Similarity', color='white', fontsize=13, pad=12)
    ax.tick_params(colors='#aaaaaa')
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_color('#444444')
    ax.spines['left'].set_color('#444444')
    for label in ax.get_yticklabels():
        label.set_color('white')

    plt.tight_layout()
    plt.show()

    # Feedback
    worst_part = min(body, key=lambda p: body[p]['score'])
    best_part = max(body, key=lambda p: body[p]['score'])
    print(f'\n💪 Best part:    {best_part}  ({body[best_part]["score"]:.0%})')
    print(f'🔧 Needs improvement: {worst_part} ({body[worst_part]["score"]:.0%})')
else:
    print('กรุณารัน Cell 7 ก่อน')

In [ ]:
# Cell 10: ดู Comparison Video (ถ้าสร้างสำเร็จ)
if 'results' in dir() and results.get('output_video') and os.path.exists(results['output_video']):
    print(f'Video saved: {results["output_video"]}')
    display(Video(results['output_video'], width=800, embed=True))
else:
    print('Comparison video ไม่ถูกสร้าง หรือกรุณารัน Cell 7 ก่อน')

---
## 📖 Bonus: สร้าง PDF คู่มือท่าเต้น

ภาพจริงจากวิดีโอ + skeleton วาดทับ รวมเป็น PDF พิมพ์ออกมาวางข้างกระจกตอนฝึกได้เลย!

In [ ]:
# Cell 11: ตั้งค่า PDF Guide
from pose_guide_generator import generate_dance_guide

SONG_NAME  = "Up"   # 👈 แสดงบนหน้าปก PDF
MAX_POSES  = 400    # จำนวนเฟรมสูงสุด (filmstrip ได้เยอะ)
PDF_MODE   = "filmstrip"  # "hold" = หยุดนิ่ง | "motion" = ขยับ | "filmstrip" = ทุก 0.5 วิ
PDF_OUTPUT = os.path.join(OUTPUT_DIR, "up_dance_guide.pdf")

print(f'เพลง: {SONG_NAME}')
print(f'โหมด: {PDF_MODE}')
print(f'จำนวนเฟรมสูงสุด: {MAX_POSES}')
print(f'Output: {PDF_OUTPUT}')

In [ ]:
# Cell 12: สร้าง PDF คู่มือท่าเต้น 🎉
pdf_path = generate_dance_guide(
    video_path=ref_video_path,
    output_path=PDF_OUTPUT,
    person_index=PERSON_INDEX,
    max_poses=MAX_POSES,
    poses_per_row=2,
    song_name=SONG_NAME,
    mode=PDF_MODE,
)

print(f'\n✅ PDF พร้อมแล้ว! เปิดได้ที่: {pdf_path}')
print('   → พิมพ์ออกมาวางข้างกระจกตอนฝึกได้เลยครับ 🎵')

---
## 🔄 วิธีใช้ซ้ำกับวิดีโอใหม่ (ไม่ต้อง extract reference ใหม่)

หลังจาก extract reference แล้วครั้งแรก สามารถโหลด cache และเปรียบเทียบกับวิดีโอใหม่ได้เลย

In [ ]:
# Cell 11: โหลด cached reference และเปรียบเทียบวิดีโอใหม่

# โหลด reference ที่ extract ไว้แล้ว
# ref_data_cached = load_pose_data(os.path.join(OUTPUT_DIR, 'pose_cache', 'reference'))

# เปรียบเทียบกับวิดีโอใหม่
# new_video = "my_dance_take2.mp4"
# results2 = compare_dance(ref_data_cached, new_video, output_dir=OUTPUT_DIR)

print('Uncomment โค้ดด้านบนเพื่อเปรียบเทียบกับวิดีโอใหม่')

---
## 💡 Tips สำหรับผลลัพธ์ที่ดีขึ้น

**เพิ่ม accuracy:**
- ใช้ `SAMPLE_FPS = 15` หรือ `20` เพื่อวิเคราะห์ละเอียดขึ้น (ช้าลงบ้าง)
- ตัดช่วงท่าเต้นที่ต้องการจาก YouTube ให้ตรงกัน

**ถ้า Detection Rate ต่ำ (<60%):**
- ตรวจสอบแสงในวิดีโอ
- ให้เห็นทั้งตัวในกล้อง
- ลด `min_detection_confidence` ในฟังก์ชัน `extract_pose_from_video`

**K-pop Songs ที่ทดสอบได้ดี:**
- เพลงที่มี Dance Practice (ห้องซ้อม) มักให้ผลดีที่สุด
- MV ที่กล้องไม่เปลี่ยนเร็วมาก
- ระวัง MV ที่มีหลายคนเต้นพร้อมกัน (จะ detect คนที่อยู่หน้าสุด)